In [ ]:

import base64, json, os, pathlib, subprocess, sys, time, torch
T0 = time.monotonic(); CAP = 2400.0; PER_SEED = 700.0
w = pathlib.Path("/kaggle/working"); p = w / "payload"; p.mkdir(exist_ok=True)
BLOB = {'heat_lbfgs_uniform_runner.py': 'IiIiRVhQLTAwMTI6IEwtQkZHUyBvbiB0aGUgQ09OVFJPTCdzIHVuaWZvcm0gcG9vbCwgc3RhcnRpbmcgZnJvbSB0aGUgUkFEIGVuZC1vZi1BZGFtIGNoZWNrcG9pbnRzIG9mIEVYUC0wMDExIChubyByZXRyYWluaW5nKS4gQWRhcHRlZCBmcm9tIGhlYXRfbGJmZ3NfZ3B1LnB5Lg0KT1JJR0lOQUw6IEdQVSByZWRvIG9mIHRoZSBIZWF0IEwtQkZHUyBzdGFnZSAoc2FtZSBhbGdvcml0aG0gYXMgaGVhdF9sYmZnc19yZWRvLnB5LCBDVURBIGZsb2F0NjQsIHBlci1zZWVkIHRpbWUgY2FwKS4NClN0YXJ0cyBmcm9tIHRoZSBhcmNoaXZlZCBTMiBjaGVja3BvaW50LWFkYW0tZmluYWwucHQgb2Ygc2VlZHMgOTAxLTkwNSArIHRoZWlyIHNhdmVkIHBvb2xzLiBDaHVuayBtYXhfZXZhbD0xMjU7IHN0b3BzIG9uDQp6ZXJvLXByb2dyZXNzIGNodW5rLCBwbGF0ZWF1LCBpdGVyYXRpb24gY2FwIDE1MDAwIG9yIC0tc2Vjb25kcy4gU2NvcmVkIG9uIGFuYWx5dGljIDUxMngyNTYgKHNhbWUgZm9ybXVsYSBhcyBhcmNoaXZlZCBzY29yZXIpLg0KVXNhZ2U6IHB5dGhvbiBoZWF0X2xiZmdzX2dwdS5weSBTRUVEIC0tc2Vjb25kcyA3MDAgLS1kaXIgL2thZ2dsZS93b3JraW5nL3BheWxvYWQgLS1vdXQgL2thZ2dsZS93b3JraW5nL3Jlc3VsdHMiIiINCmltcG9ydCBhcmdwYXJzZSwganNvbiwgdGltZQ0KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoDQppbXBvcnQgbnVtcHkgYXMgbnAsIHRvcmNoDQoNCmFwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKTsgYXAuYWRkX2FyZ3VtZW50KCJzZWVkIiwgdHlwZT1pbnQpOyBhcC5hZGRfYXJndW1lbnQoIi0tc2Vjb25kcyIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9NzAwLjApDQphcC5hZGRfYXJndW1lbnQoIi0tZGlyIiwgZGVmYXVsdD0icGF5bG9hZCIpOyBhcC5hZGRfYXJndW1lbnQoIi0tb3V0IiwgZGVmYXVsdD0icmVzdWx0cyIpOyBhcC5hZGRfYXJndW1lbnQoIi0tZGV2aWNlIiwgZGVmYXVsdD0iY3VkYTowIik7IGEgPSBhcC5wYXJzZV9hcmdzKCkNCmltcG9ydCBzeXM7IHN5cy5wYXRoLmluc2VydCgwLCBhLmRpcikNCmZyb20gYWZfc2NpbWxfbW9kZWwgaW1wb3J0IGJ1aWxkX21vZGVsDQp0b3JjaC5zZXRfZGVmYXVsdF9kdHlwZSh0b3JjaC5mbG9hdDY0KQ0KZGV2ID0gdG9yY2guZGV2aWNlKGEuZGV2aWNlKQ0KQUxQSEEsIFJTLCBNQVhJVEVSLCBDSFVOSywgTUFYRVZBTCA9IDAuMSwgMS4xLCAxNTAwMCwgMjUsIDEyNQ0KUyA9IGEuc2VlZDsgb3V0ID0gUGF0aChhLm91dCkgLyBmInNlZWQte1N9Ijsgb3V0Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkNCmNrID0gdG9yY2gubG9hZChQYXRoKGEuZGlyKSAvIGYic2VlZC17U30tYWRhbS1maW5hbC5wdCIsIG1hcF9sb2NhdGlvbj0iY3B1Iiwgd2VpZ2h0c19vbmx5PUZhbHNlKQ0KYXNzZXJ0IGNrWyJzdGFnZSJdID09ICJhZGFtX2ZpbmFsX3JhZCIsIGNrWyJzdGFnZSJdDQptb2RlbCA9IGJ1aWxkX21vZGVsKCJoZWF0IiwgeyJpZCI6ICJyYmZfZml4ZWQifSk7IG1vZGVsLmxvYWRfc3RhdGVfZGljdChja1sic3RhdGVfZGljdCJdKTsgbW9kZWwgPSBtb2RlbC50byhkZXYpLmRvdWJsZSgpDQp6ID0gbnAubG9hZChQYXRoKGEuZGlyKSAvIGYiaGVhdC1zZWVkLXtTfS5ucHoiKQ0KcHRzID0gdG9yY2gudGVuc29yKHpbImludGVyaW9yIl0sIGR0eXBlPXRvcmNoLmZsb2F0NjQsIGRldmljZT1kZXYsIHJlcXVpcmVzX2dyYWQ9VHJ1ZSkNCg0KDQpkZWYgdHJhcChuKToNCiAgICB3ID0gdG9yY2gub25lcyhuLCBkdHlwZT10b3JjaC5mbG9hdDY0LCBkZXZpY2U9ZGV2KTsgd1swXSA9IHdbLTFdID0gMC41OyByZXR1cm4gdyAvIChuIC0gMSkNCg0KDQpkZWYgbG9zc19mbigpOg0KICAgIG8gPSBtb2RlbChwdHMpDQogICAgZyA9IHRvcmNoLmF1dG9ncmFkLmdyYWQobywgcHRzLCB0b3JjaC5vbmVzX2xpa2UobyksIGNyZWF0ZV9ncmFwaD1UcnVlLCByZXRhaW5fZ3JhcGg9VHJ1ZSlbMF0NCiAgICB1eHggPSB0b3JjaC5hdXRvZ3JhZC5ncmFkKGdbOiwgMDoxXSwgcHRzLCB0b3JjaC5vbmVzX2xpa2UoZ1s6LCAwOjFdKSwgY3JlYXRlX2dyYXBoPVRydWUsIHJldGFpbl9ncmFwaD1UcnVlKVswXVs6LCAwOjFdDQogICAgcmV0dXJuIHRvcmNoLm1lYW4oKChnWzosIDE6Ml0gLSBBTFBIQSAqIHV4eCkgLyBSUykuc3F1YXJlKCkpDQoNCg0KZGVmIHNjb3JlKCk6DQogICAgbngsIG50ID0gNTEyLCAyNTYNCiAgICB4ID0gdG9yY2gubGluc3BhY2UoMCwgMSwgbngsIGRldmljZT1kZXYpOyB0ID0gdG9yY2gubGluc3BhY2UoMCwgMSwgbnQsIGRldmljZT1kZXYpDQogICAgeHgsIHR0ID0gdG9yY2gubWVzaGdyaWQoeCwgdCwgaW5kZXhpbmc9ImlqIikNCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToNCiAgICAgICAgcHJlZCA9IG1vZGVsKHRvcmNoLnN0YWNrKCh4eC5yZXNoYXBlKC0xKSwgdHQucmVzaGFwZSgtMSkpLCAxKSkucmVzaGFwZShueCwgbnQpDQogICAgZXggPSB0b3JjaC5zaW4odG9yY2gucGkgKiB4eCkgKiB0b3JjaC5leHAoLUFMUEhBICogdG9yY2gucGkgKiogMiAqIHR0KQ0KICAgIHcgPSB0cmFwKG54KVs6LCBOb25lXSAqIHRyYXAobnQpW05vbmUsIDpdDQogICAgcmV0dXJuIGZsb2F0KHRvcmNoLnNxcnQodG9yY2guc3VtKHcgKiAocHJlZCAtIGV4KSAqKiAyKSAvIHRvcmNoLnN1bSh3ICogZXggKiogMikpKQ0KDQoNCm9wdCA9IHRvcmNoLm9wdGltLkxCRkdTKG1vZGVsLnBhcmFtZXRlcnMoKSwgbHI9MS4wLCBtYXhfaXRlcj1DSFVOSywgbWF4X2V2YWw9TUFYRVZBTCwgdG9sZXJhbmNlX2dyYWQ9MWUtOCwNCiAgICAgICAgICAgICAgICAgICAgICAgIHRvbGVyYW5jZV9jaGFuZ2U9MC4wLCBoaXN0b3J5X3NpemU9MTAwLCBsaW5lX3NlYXJjaF9mbj0ic3Ryb25nX3dvbGZlIikNCmV2YWxzID0gMA0KDQoNCmRlZiBjbG9zdXJlKCk6DQogICAgZ2xvYmFsIGV2YWxzDQogICAgb3B0Lnplcm9fZ3JhZChzZXRfdG9fbm9uZT1UcnVlKTsgdiA9IGxvc3NfZm4oKTsgdi5iYWNrd2FyZCgpOyBldmFscyArPSAxOyByZXR1cm4gdg0KDQoNCih0b3JjaC5jdWRhLnN5bmNocm9uaXplKCkgaWYgZGV2LnR5cGUgPT0gImN1ZGEiIGVsc2UgTm9uZSk7IHQwID0gdGltZS5wZXJmX2NvdW50ZXIoKQ0KbGFzdCA9IGZsb2F0KGxvc3NfZm4oKS5kZXRhY2goKSk7IGN1cnZlID0gW3siaXRlcnMiOiAwLCAibG9zcyI6IGxhc3QsICJyZWxfbDIiOiBzY29yZSgpLCAiZXZhbHMiOiAwLCAic2VjIjogMC4wfV0NCnByaW50KGpzb24uZHVtcHMoY3VydmVbMF0pLCBmbHVzaD1UcnVlKQ0KaXRlcnMsIHBsYXRlYXUsIHJlYXNvbiA9IDAsIDAsICJtYXhfaXRlcmF0aW9ucyI7IG1hcmtzID0gWzUwLCAyMDAsIDEwMDAsIDMwMDAsIDc1MDAsIDE1MDAwXQ0Kd2hpbGUgaXRlcnMgPCBNQVhJVEVSOg0KICAgIGlmIHRpbWUucGVyZl9jb3VudGVyKCkgLSB0MCA+PSBhLnNlY29uZHM6IHJlYXNvbiA9ICJ0aW1lX2NhcCI7IGJyZWFrDQogICAgYmVmb3JlID0gaW50KG9wdC5zdGF0ZVtvcHQuX3BhcmFtc1swXV0uZ2V0KCJuX2l0ZXIiLCAwKSk7IG9wdC5zdGVwKGNsb3N1cmUpDQogICAgYWZ0ZXIgPSBpbnQob3B0LnN0YXRlW29wdC5fcGFyYW1zWzBdXS5nZXQoIm5faXRlciIsIGJlZm9yZSkpOyBkID0gYWZ0ZXIgLSBiZWZvcmU7IGl0ZXJzICs9IGQNCiAgICBjdXIgPSBmbG9hdChsb3NzX2ZuKCkuZGV0YWNoKCkpDQogICAgaWYgbm90IG5wLmlzZmluaXRlKGN1cik6IHJlYXNvbiA9ICJub25maW5pdGUiOyBicmVhaw0KICAgIGlmIGQgPT0gMDogcmVhc29uID0gIm5vX3Byb2dyZXNzX2NodW5rIjsgYnJlYWsNCiAgICBwbGF0ZWF1ID0gcGxhdGVhdSArIDEgaWYgKGxhc3QgLSBjdXIpIC8gbWF4KGFicyhsYXN0KSwgMWUtMzAwKSA8IDFlLTEwIGVsc2UgMDsgbGFzdCA9IGN1cg0KICAgIGlmIGFueShpdGVycyA+PSBtIGFuZCBjdXJ2ZVstMV1bIml0ZXJzIl0gPCBtIGZvciBtIGluIG1hcmtzKToNCiAgICAgICAgY3VydmUuYXBwZW5kKHsiaXRlcnMiOiBpdGVycywgImxvc3MiOiBjdXIsICJyZWxfbDIiOiBzY29yZSgpLCAiZXZhbHMiOiBldmFscywgInNlYyI6IHRpbWUucGVyZl9jb3VudGVyKCkgLSB0MH0pDQogICAgICAgIHByaW50KGpzb24uZHVtcHMoY3VydmVbLTFdKSwgZmx1c2g9VHJ1ZSkNCiAgICBpZiBwbGF0ZWF1ID49IDU6IHJlYXNvbiA9ICJsb3NzX3BsYXRlYXUiOyBicmVhaw0KKHRvcmNoLmN1ZGEuc3luY2hyb25pemUoKSBpZiBkZXYudHlwZSA9PSAiY3VkYSIgZWxzZSBOb25lKQ0KZmluYWwgPSB7InNlZWQiOiBTLCAiaXRlcnMiOiBpdGVycywgImV2YWxzIjogZXZhbHMsICJzdG9wX3JlYXNvbiI6IHJlYXNvbiwgImxvc3MiOiBsYXN0LCAicmVsX2wyXzUxMngyNTYiOiBzY29yZSgpLA0KICAgICAgICAgImxiZmdzX3NlY29uZHMiOiB0aW1lLnBlcmZfY291bnRlcigpIC0gdDAsICJhZGFtX2ZpbmFsX3JlbF9sMiI6IGN1cnZlWzBdWyJyZWxfbDIiXSwgImN1cnZlIjogY3VydmUsDQogICAgICAgICAicG9vbF9zaGEyNTYiOiBzdHIoelsicG9vbF9zaGEyNTYiXSksICJkZXZpY2UiOiAodG9yY2guY3VkYS5nZXRfZGV2aWNlX25hbWUoMCkgaWYgZGV2LnR5cGUgPT0gImN1ZGEiIGVsc2UgImNwdSIpLCAicG9vbCI6ICJjb250cm9sIHVuaWZvcm0gcG9vbCAoRVhQLTAwMDgpIGhlYXQtc2VlZC1TLm5weiBpbnRlcmlvciIsICJleHBlcmltZW50IjogIkVYUC0wMDEyIiwgInNjaWVudGlmaWNfYWNjZXB0ZWQiOiBGYWxzZX0NCnRvcmNoLnNhdmUoeyJzdGF0ZV9kaWN0IjogbW9kZWwuc3RhdGVfZGljdCgpLCAic3RhZ2UiOiAibGJmZ3NfdW5pZm9ybV9hZnRlcl9yYWQiLCAiaXRlcnMiOiBpdGVyc30sIG91dCAvICJjaGVja3BvaW50LWxiZmdzLXVuaWZvcm0ucHQiKQ0KanNvbi5kdW1wKGZpbmFsLCBvcGVuKG91dCAvICJyZXN1bHQuanNvbiIsICJ3IiksIGluZGVudD0yKQ0KcHJpbnQoIkZJTkFMIiwganNvbi5kdW1wcyh7azogdiBmb3IgaywgdiBpbiBmaW5hbC5pdGVtcygpIGlmIGsgIT0gImN1cnZlIn0pLCBmbHVzaD1UcnVlKQ0K', 'af_sciml_model.py': 'IiIiUGFyYW1ldGVyLW1hdGNoZWQgbmV1cmFsIGFuZCBLb2xtb2dvcm92LUFybm9sZCBtb2RlbHMgZm9yIFNjaU1MIHNjcmVlbmluZy4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCB0b3JjaApmcm9tIHRvcmNoIGltcG9ydCBubgpmcm9tIHRvcmNoLm5uIGltcG9ydCBmdW5jdGlvbmFsIGFzIEYKCgpjbGFzcyBSQkZMYXllcihubi5Nb2R1bGUpOgogICAgIiIiRml4ZWQgR2F1c3NpYW4gcmFkaWFsIGJhc2VzIHBsdXMgYSB0cmFpbmFibGUgU2lMVSBlZGdlIGZ1bmN0aW9uLiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBpbnB1dHM6IGludCwgb3V0cHV0czogaW50LCBiYXNpc19jb3VudDogaW50ID0gMTEpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYucmVnaXN0ZXJfYnVmZmVyKCJjZW50ZXJzIiwgdG9yY2gubGluc3BhY2UoLTEuMDUsIDEuMDUsIGJhc2lzX2NvdW50KSkKICAgICAgICBzZWxmLnJlZ2lzdGVyX2J1ZmZlcigid2lkdGgiLCB0b3JjaC50ZW5zb3IoMC4yMSkpCiAgICAgICAgc2VsZi5jb2VmZmljaWVudHMgPSBubi5QYXJhbWV0ZXIodG9yY2guZW1wdHkob3V0cHV0cywgaW5wdXRzLCBiYXNpc19jb3VudCkpCiAgICAgICAgc2VsZi5iYXNlID0gbm4uUGFyYW1ldGVyKHRvcmNoLmVtcHR5KG91dHB1dHMsIGlucHV0cykpCiAgICAgICAgbm4uaW5pdC5ub3JtYWxfKHNlbGYuY29lZmZpY2llbnRzLCBzdGQ9MC4xIC8gKGlucHV0cyoqMC41KSkKICAgICAgICBubi5pbml0Lm5vcm1hbF8oc2VsZi5iYXNlLCBzdGQ9MC4xIC8gKGlucHV0cyoqMC41KSkKCiAgICBkZWYgYmFzaXMoc2VsZiwgeDogdG9yY2guVGVuc29yKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgcmV0dXJuIHRvcmNoLmV4cCgtKCh4LnVuc3F1ZWV6ZSgtMSkgLSBzZWxmLmNlbnRlcnMpIC8gc2VsZi53aWR0aCkgKiogMikKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4OiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICByZXR1cm4gdG9yY2guZWluc3VtKCJiaWcsb2lnLT5ibyIsIHNlbGYuYmFzaXMoeCksIHNlbGYuY29lZmZpY2llbnRzKSArIEYubGluZWFyKAogICAgICAgICAgICBGLnNpbHUoeCksIHNlbGYuYmFzZQogICAgICAgICkKCgpjbGFzcyBCU3BsaW5lTGF5ZXIobm4uTW9kdWxlKToKICAgICIiIkN1YmljIG9wZW4tdW5pZm9ybSBCLXNwbGluZSBlZGdlIGZ1bmN0aW9ucyB3aXRoIGZpeGVkIGtub3RzLiIiIgoKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIGlucHV0czogaW50LAogICAgICAgIG91dHB1dHM6IGludCwKICAgICAgICBiYXNpc19jb3VudDogaW50ID0gMTEsCiAgICAgICAgZGVncmVlOiBpbnQgPSAzLAogICAgICAgIGxvdzogZmxvYXQgPSAtMS4wNSwKICAgICAgICBoaWdoOiBmbG9hdCA9IDEuMDUsCiAgICApOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIGlmIGJhc2lzX2NvdW50IDw9IGRlZ3JlZToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiYmFzaXNfY291bnQgbXVzdCBleGNlZWQgc3BsaW5lIGRlZ3JlZSIpCiAgICAgICAgaWYgbm90IGxvdyA8IGhpZ2g6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInNwbGluZSBpbnRlcnZhbCBtdXN0IGhhdmUgcG9zaXRpdmUgd2lkdGgiKQogICAgICAgIGludGVyaW9yX2NvdW50ID0gYmFzaXNfY291bnQgLSBkZWdyZWUgLSAxCiAgICAgICAgaW50ZXJpb3IgPSAoCiAgICAgICAgICAgIHRvcmNoLmxpbnNwYWNlKGxvdywgaGlnaCwgaW50ZXJpb3JfY291bnQgKyAyKVsxOi0xXQogICAgICAgICAgICBpZiBpbnRlcmlvcl9jb3VudAogICAgICAgICAgICBlbHNlIHRvcmNoLmVtcHR5KDApCiAgICAgICAgKQogICAgICAgIGtub3RzID0gdG9yY2guY2F0KAogICAgICAgICAgICAoCiAgICAgICAgICAgICAgICB0b3JjaC5mdWxsKChkZWdyZWUgKyAxLCksIGxvdyksCiAgICAgICAgICAgICAgICBpbnRlcmlvciwKICAgICAgICAgICAgICAgIHRvcmNoLmZ1bGwoKGRlZ3JlZSArIDEsKSwgaGlnaCksCiAgICAgICAgICAgICkKICAgICAgICApCiAgICAgICAgc2VsZi5yZWdpc3Rlcl9idWZmZXIoImtub3RzIiwga25vdHMpCiAgICAgICAgc2VsZi5kZWdyZWUgPSBkZWdyZWUKICAgICAgICBzZWxmLmNvZWZmaWNpZW50cyA9IG5uLlBhcmFtZXRlcih0b3JjaC5lbXB0eShvdXRwdXRzLCBpbnB1dHMsIGJhc2lzX2NvdW50KSkKICAgICAgICBzZWxmLmJhc2UgPSBubi5QYXJhbWV0ZXIodG9yY2guZW1wdHkob3V0cHV0cywgaW5wdXRzKSkKICAgICAgICBubi5pbml0Lm5vcm1hbF8oc2VsZi5jb2VmZmljaWVudHMsIHN0ZD0wLjEgLyAoaW5wdXRzKiowLjUpKQogICAgICAgIG5uLmluaXQubm9ybWFsXyhzZWxmLmJhc2UsIHN0ZD0wLjEgLyAoaW5wdXRzKiowLjUpKQoKICAgIGRlZiBiYXNpcyhzZWxmLCB4OiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICAiIiJFdmFsdWF0ZSB0aGUgQ294LWRlIEJvb3IgYmFzaXMsIHJldGFpbmluZyBpbnB1dCBhdXRvZ3JhZCBkZXJpdmF0aXZlcy4iIiIKICAgICAgICBjb29yZGluYXRlID0geC51bnNxdWVlemUoLTEpCiAgICAgICAgYmFzaXMgPSAoKGNvb3JkaW5hdGUgPj0gc2VsZi5rbm90c1s6LTFdKSAmIChjb29yZGluYXRlIDwgc2VsZi5rbm90c1sxOl0pKS50byh4LmR0eXBlKQogICAgICAgICMgSW5wdXRzIGluIHRoaXMgZXhwZXJpbWVudCBsaWUgc3RyaWN0bHkgaW5zaWRlIHRoZSBwYWRkZWQga25vdCBpbnRlcnZhbC4KICAgICAgICAjIEtlZXBpbmcgdGhlIGVuZHBvaW50IGNvbnZlbnRpb24gbWFrZXMgdGhlIGxheWVyIHdlbGwtZGVmaW5lZCBpZiB0aGUKICAgICAgICAjIGludGVydmFsIGlzIGxhdGVyIHVzZWQgZGlyZWN0bHkgYXMgYSBtb2RlbCBkb21haW4uCiAgICAgICAgYXRfcmlnaHRfZW5kcG9pbnQgPSBjb29yZGluYXRlLnNxdWVlemUoLTEpID09IHNlbGYua25vdHNbLTFdCiAgICAgICAgaWYgYXRfcmlnaHRfZW5kcG9pbnQuYW55KCk6CiAgICAgICAgICAgIGVuZHBvaW50ID0gdG9yY2guemVyb3NfbGlrZShiYXNpcykKICAgICAgICAgICAgZW5kcG9pbnRbLi4uLCAtMV0gPSBhdF9yaWdodF9lbmRwb2ludC50byh4LmR0eXBlKQogICAgICAgICAgICBiYXNpcyA9IHRvcmNoLndoZXJlKGF0X3JpZ2h0X2VuZHBvaW50LnVuc3F1ZWV6ZSgtMSksIGVuZHBvaW50LCBiYXNpcykKCiAgICAgICAgZm9yIG9yZGVyIGluIHJhbmdlKDEsIHNlbGYuZGVncmVlICsgMSk6CiAgICAgICAgICAgIGxlZnRfZGVub21pbmF0b3IgPSBzZWxmLmtub3RzW29yZGVyOi0xXSAtIHNlbGYua25vdHNbOiAtKG9yZGVyICsgMSldCiAgICAgICAgICAgIHJpZ2h0X2Rlbm9taW5hdG9yID0gc2VsZi5rbm90c1tvcmRlciArIDEgOl0gLSBzZWxmLmtub3RzWzE6LW9yZGVyXQogICAgICAgICAgICBsZWZ0X3dlaWdodCA9IChjb29yZGluYXRlIC0gc2VsZi5rbm90c1s6IC0ob3JkZXIgKyAxKV0pIC8gbGVmdF9kZW5vbWluYXRvci5jbGFtcF9taW4oCiAgICAgICAgICAgICAgICB0b3JjaC5maW5mbyh4LmR0eXBlKS5lcHMKICAgICAgICAgICAgKQogICAgICAgICAgICByaWdodF93ZWlnaHQgPSAoc2VsZi5rbm90c1tvcmRlciArIDEgOl0gLSBjb29yZGluYXRlKSAvIHJpZ2h0X2Rlbm9taW5hdG9yLmNsYW1wX21pbigKICAgICAgICAgICAgICAgIHRvcmNoLmZpbmZvKHguZHR5cGUpLmVwcwogICAgICAgICAgICApCiAgICAgICAgICAgIGxlZnRfd2VpZ2h0ID0gdG9yY2gud2hlcmUobGVmdF9kZW5vbWluYXRvciA+IDAsIGxlZnRfd2VpZ2h0LCAwLjApCiAgICAgICAgICAgIHJpZ2h0X3dlaWdodCA9IHRvcmNoLndoZXJlKHJpZ2h0X2Rlbm9taW5hdG9yID4gMCwgcmlnaHRfd2VpZ2h0LCAwLjApCiAgICAgICAgICAgIGJhc2lzID0gbGVmdF93ZWlnaHQgKiBiYXNpc1suLi4sIDotMV0gKyByaWdodF93ZWlnaHQgKiBiYXNpc1suLi4sIDE6XQogICAgICAgIHJldHVybiBiYXNpcwoKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHg6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgICAgIHJldHVybiB0b3JjaC5laW5zdW0oImJpZyxvaWctPmJvIiwgc2VsZi5iYXNpcyh4KSwgc2VsZi5jb2VmZmljaWVudHMpICsgRi5saW5lYXIoCiAgICAgICAgICAgIEYuc2lsdSh4KSwgc2VsZi5iYXNlCiAgICAgICAgKQoKCmNsYXNzIF9LQU5CYXNlKG5uLk1vZHVsZSk6CiAgICBtb2RlbF9mYW1pbHk6IHN0cgoKICAgIGRlZiBub3JtYWxpemVkKHNlbGYsIHg6IHRvcmNoLlRlbnNvcikgLT4gdG9yY2guVGVuc29yOgogICAgICAgIGlmIGhhc2F0dHIoeCwgInRlbnNvciIpOgogICAgICAgICAgICB4ID0geC50ZW5zb3IKICAgICAgICB4X3NwYWNlID0gMi4wICogeFs6LCAwOjFdIC0gMS4wIGlmIHNlbGYucHJvYmxlbSA9PSAiaGVhdCIgZWxzZSB4WzosIDA6MV0KICAgICAgICByZXR1cm4gdG9yY2guY2F0KCh4X3NwYWNlLCAyLjAgKiB4WzosIDE6Ml0gLSAxLjApLCBkaW09MSkKCiAgICBkZWYgY29uc3RyYWluKHNlbGYsIHg6IHRvcmNoLlRlbnNvciwgcmF3OiB0b3JjaC5UZW5zb3IpIC0+IHRvcmNoLlRlbnNvcjoKICAgICAgICBpZiBzZWxmLnByb2JsZW0gPT0gImhlYXQiOgogICAgICAgICAgICByZXR1cm4gdG9yY2guc2luKHRvcmNoLnBpICogeFs6LCAwOjFdKSArIHhbOiwgMToyXSAqIHhbOiwgMDoxXSAqICgxLjAgLSB4WzosIDA6MV0pICogcmF3CiAgICAgICAgaWYgc2VsZi5wcm9ibGVtID09ICJhbGxlbl9jYWhuIjoKICAgICAgICAgICAgc3BhY2UgPSB4WzosIDA6MV0KICAgICAgICAgICAgdGltZSA9IHhbOiwgMToyXQogICAgICAgICAgICBpbml0aWFsID0gc3BhY2Uuc3F1YXJlKCkgKiB0b3JjaC5jb3ModG9yY2gucGkgKiBzcGFjZSkKICAgICAgICAgICAgcmV0dXJuIGluaXRpYWwgKyB0aW1lICogKDEuMCAtIHNwYWNlLnNxdWFyZSgpKSAqIHJhdwogICAgICAgIHJldHVybiByYXcKCiAgICBkZWYgZm9yd2FyZF93aXRoX3RyYWNlKAogICAgICAgIHNlbGYsIHg6IHRvcmNoLlRlbnNvcgogICAgKSAtPiB0dXBsZVt0b3JjaC5UZW5zb3IsIGxpc3RbZGljdFtzdHIsIHRvcmNoLlRlbnNvcl1dXToKICAgICAgICBpZiBoYXNhdHRyKHgsICJ0ZW5zb3IiKToKICAgICAgICAgICAgeCA9IHgudGVuc29yCiAgICAgICAgaCA9IHNlbGYubm9ybWFsaXplZCh4KQogICAgICAgIHRyYWNlczogbGlzdFtkaWN0W3N0ciwgdG9yY2guVGVuc29yXV0gPSBbXQogICAgICAgIGZvciBpbmRleCwgbGF5ZXIgaW4gZW51bWVyYXRlKHNlbGYubGF5ZXJzKToKICAgICAgICAgICAgdHJhY2U6IGRpY3Rbc3RyLCB0b3JjaC5UZW5zb3JdID0ge30KICAgICAgICAgICAgaWYgaGFzYXR0cihsYXllciwgImJhc2lzIik6CiAgICAgICAgICAgICAgICB0cmFjZVsiYmFzaXMiXSA9IGxheWVyLmJhc2lzKGgpCiAgICAgICAgICAgIHByZWFjdGl2YXRpb24gPSBsYXllcihoKQogICAgICAgICAgICB0cmFjZVsicHJlYWN0aXZhdGlvbiJdID0gcHJlYWN0aXZhdGlvbgogICAgICAgICAgICBoID0gdG9yY2gudGFuaChwcmVhY3RpdmF0aW9uKSBpZiBpbmRleCA8IGxlbihzZWxmLmxheWVycykgLSAxIGVsc2UgcHJlYWN0aXZhdGlvbgogICAgICAgICAgICB0cmFjZVsiYWN0aXZhdGlvbiJdID0gaAogICAgICAgICAgICB0cmFjZXMuYXBwZW5kKHRyYWNlKQogICAgICAgIHJldHVybiBzZWxmLmNvbnN0cmFpbih4LCBoKSwgdHJhY2VzCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeDogdG9yY2guVGVuc29yKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgb3V0cHV0LCBfID0gc2VsZi5mb3J3YXJkX3dpdGhfdHJhY2UoeCkKICAgICAgICByZXR1cm4gb3V0cHV0CgoKY2xhc3MgUkJGUElLQU4oX0tBTkJhc2UpOgogICAgIiIiRml4ZWQtY2VudGVyIFJCRiBLQU4sIHJldGFpbmVkIGFzIHRoZSBleGlzdGluZyBtb2RlbCBjb250cm9sLiIiIgoKICAgIG1vZGVsX2lkID0gInJiZl9maXhlZCIKICAgIG1vZGVsX2ZhbWlseSA9ICJyYmYiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHByb2JsZW06IHN0ciwgYmFzaXNfY291bnQ6IGludCA9IDExKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBpZiBwcm9ibGVtIG5vdCBpbiAoImhlYXQiLCAiYnVyZ2VycyIsICJhbGxlbl9jYWhuIik6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInVua25vd24gcHJvYmxlbSIpCiAgICAgICAgc2VsZi5wcm9ibGVtID0gcHJvYmxlbQogICAgICAgIHNlbGYubGF5ZXJzID0gbm4uTW9kdWxlTGlzdCgKICAgICAgICAgICAgW1JCRkxheWVyKGEsIGIsIGJhc2lzX2NvdW50KSBmb3IgYSwgYiBpbiB6aXAoWzIsIDE2LCAxNl0sIFsxNiwgMTYsIDFdKV0KICAgICAgICApCgoKY2xhc3MgQlNwbGluZVBJS0FOKF9LQU5CYXNlKToKICAgICIiIkZpeGVkIGN1YmljIEItc3BsaW5lIEtBTiB1c2luZyB0aGUgc2FtZSBlZGdlIGFuZCBsYXllciBkaW1lbnNpb25zLiIiIgoKICAgIG1vZGVsX2lkID0gImJzcGxpbmVfY3ViaWMiCiAgICBtb2RlbF9mYW1pbHkgPSAiYnNwbGluZSIKCiAgICBkZWYgX19pbml0X18oc2VsZiwgcHJvYmxlbTogc3RyLCBiYXNpc19jb3VudDogaW50ID0gMTEsIGRlZ3JlZTogaW50ID0gMyk6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgaWYgcHJvYmxlbSBub3QgaW4gKCJoZWF0IiwgImJ1cmdlcnMiLCAiYWxsZW5fY2FobiIpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJ1bmtub3duIHByb2JsZW0iKQogICAgICAgIHNlbGYucHJvYmxlbSA9IHByb2JsZW0KICAgICAgICBzZWxmLmxheWVycyA9IG5uLk1vZHVsZUxpc3QoCiAgICAgICAgICAgIFsKICAgICAgICAgICAgICAgIEJTcGxpbmVMYXllcihhLCBiLCBiYXNpc19jb3VudCwgZGVncmVlKQogICAgICAgICAgICAgICAgZm9yIGEsIGIgaW4gemlwKFsyLCAxNiwgMTZdLCBbMTYsIDE2LCAxXSkKICAgICAgICAgICAgXQogICAgICAgICkKCgpjbGFzcyBQSU5OTUxQKF9LQU5CYXNlKToKICAgICIiIlRhbmggUElOTiB3aXRoIGEgaGlkZGVuIHdpZHRoIGNob3NlbiB0byBtYXRjaCB0aGUgUkJGIHBhcmFtZXRlciBjb3VudC4iIiIKCiAgICBtb2RlbF9pZCA9ICJtbHBfdGFuaCIKICAgIG1vZGVsX2ZhbWlseSA9ICJtbHAiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHByb2JsZW06IHN0ciwgd2lkdGg6IGludCA9IDU4KToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBpZiBwcm9ibGVtIG5vdCBpbiAoImhlYXQiLCAiYnVyZ2VycyIsICJhbGxlbl9jYWhuIik6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInVua25vd24gcHJvYmxlbSIpCiAgICAgICAgc2VsZi5wcm9ibGVtID0gcHJvYmxlbQogICAgICAgIHNlbGYud2lkdGggPSB3aWR0aAogICAgICAgIHNlbGYubGF5ZXJzID0gbm4uTW9kdWxlTGlzdCgKICAgICAgICAgICAgW25uLkxpbmVhcigyLCB3aWR0aCksIG5uLkxpbmVhcih3aWR0aCwgd2lkdGgpLCBubi5MaW5lYXIod2lkdGgsIDEpXQogICAgICAgICkKICAgICAgICBmb3IgbGF5ZXIgaW4gc2VsZi5sYXllcnNbOi0xXToKICAgICAgICAgICAgbm4uaW5pdC54YXZpZXJfdW5pZm9ybV8obGF5ZXIud2VpZ2h0LCBnYWluPW5uLmluaXQuY2FsY3VsYXRlX2dhaW4oInRhbmgiKSkKICAgICAgICAgICAgbm4uaW5pdC56ZXJvc18obGF5ZXIuYmlhcykKICAgICAgICBubi5pbml0Lnhhdmllcl91bmlmb3JtXyhzZWxmLmxheWVyc1stMV0ud2VpZ2h0KQogICAgICAgIG5uLmluaXQuemVyb3NfKHNlbGYubGF5ZXJzWy0xXS5iaWFzKQoKICAgIGRlZiBmb3J3YXJkX3dpdGhfdHJhY2UoCiAgICAgICAgc2VsZiwgeDogdG9yY2guVGVuc29yCiAgICApIC0+IHR1cGxlW3RvcmNoLlRlbnNvciwgbGlzdFtkaWN0W3N0ciwgdG9yY2guVGVuc29yXV1dOgogICAgICAgIGlmIGhhc2F0dHIoeCwgInRlbnNvciIpOgogICAgICAgICAgICB4ID0geC50ZW5zb3IKICAgICAgICBoID0gc2VsZi5ub3JtYWxpemVkKHgpCiAgICAgICAgdHJhY2VzOiBsaXN0W2RpY3Rbc3RyLCB0b3JjaC5UZW5zb3JdXSA9IFtdCiAgICAgICAgZm9yIGluZGV4LCBsYXllciBpbiBlbnVtZXJhdGUoc2VsZi5sYXllcnMpOgogICAgICAgICAgICBwcmVhY3RpdmF0aW9uID0gbGF5ZXIoaCkKICAgICAgICAgICAgaCA9IHRvcmNoLnRhbmgocHJlYWN0aXZhdGlvbikgaWYgaW5kZXggPCBsZW4oc2VsZi5sYXllcnMpIC0gMSBlbHNlIHByZWFjdGl2YXRpb24KICAgICAgICAgICAgdHJhY2VzLmFwcGVuZCh7InByZWFjdGl2YXRpb24iOiBwcmVhY3RpdmF0aW9uLCAiYWN0aXZhdGlvbiI6IGh9KQogICAgICAgIHJldHVybiBzZWxmLmNvbnN0cmFpbih4LCBoKSwgdHJhY2VzCgoKZGVmIGJ1aWxkX21vZGVsKHByb2JsZW06IHN0ciwgY29uZmlnOiBkaWN0IHwgTm9uZSA9IE5vbmUpIC0+IG5uLk1vZHVsZToKICAgICIiIkNyZWF0ZSBhbiBleHBsaWNpdGx5IG5hbWVkIG1vZGVsOyBSQkYgcmVtYWlucyB0aGUgYmFja3dhcmRzLWNvbXBhdGlibGUgZGVmYXVsdC4iIiIKICAgIGNvbmZpZyA9IHt9IGlmIGNvbmZpZyBpcyBOb25lIGVsc2UgZGljdChjb25maWcpCiAgICBtb2RlbF9pZCA9IGNvbmZpZy5nZXQoImlkIiwgY29uZmlnLmdldCgibW9kZWxfaWQiLCAicmJmX2ZpeGVkIikpCiAgICBpZiBtb2RlbF9pZCA9PSAicmJmX2ZpeGVkIjoKICAgICAgICByZXR1cm4gUkJGUElLQU4ocHJvYmxlbSwgYmFzaXNfY291bnQ9aW50KGNvbmZpZy5nZXQoImJhc2lzX2NvdW50IiwgMTEpKSkKICAgIGlmIG1vZGVsX2lkID09ICJic3BsaW5lX2N1YmljIjoKICAgICAgICByZXR1cm4gQlNwbGluZVBJS0FOKAogICAgICAgICAgICBwcm9ibGVtLAogICAgICAgICAgICBiYXNpc19jb3VudD1pbnQoY29uZmlnLmdldCgiYmFzaXNfY291bnQiLCAxMSkpLAogICAgICAgICAgICBkZWdyZWU9aW50KGNvbmZpZy5nZXQoImRlZ3JlZSIsIDMpKSwKICAgICAgICApCiAgICBpZiBtb2RlbF9pZCA9PSAibWxwX3RhbmgiOgogICAgICAgIHJldHVybiBQSU5OTUxQKHByb2JsZW0sIHdpZHRoPWludChjb25maWcuZ2V0KCJ3aWR0aCIsIDU4KSkpCiAgICByYWlzZSBWYWx1ZUVycm9yKGYidW5rbm93biBtb2RlbCBpZDoge21vZGVsX2lkIXJ9IikKCgpkZWYgZGVyaXZhdGl2ZXMoeTogdG9yY2guVGVuc29yLCB4OiB0b3JjaC5UZW5zb3IpIC0+IHR1cGxlW3RvcmNoLlRlbnNvciwgLi4uXToKICAgIGdyYWRpZW50ID0gdG9yY2guYXV0b2dyYWQuZ3JhZCgKICAgICAgICB5LCB4LCB0b3JjaC5vbmVzX2xpa2UoeSksIGNyZWF0ZV9ncmFwaD1UcnVlLCByZXRhaW5fZ3JhcGg9VHJ1ZQogICAgKVswXQogICAgdXggPSBncmFkaWVudFs6LCAwOjFdCiAgICB1dCA9IGdyYWRpZW50WzosIDE6Ml0KICAgIHV4eCA9IHRvcmNoLmF1dG9ncmFkLmdyYWQoCiAgICAgICAgdXgsIHgsIHRvcmNoLm9uZXNfbGlrZSh1eCksIGNyZWF0ZV9ncmFwaD1UcnVlLCByZXRhaW5fZ3JhcGg9VHJ1ZQogICAgKVswXVs6LCAwOjFdCiAgICByZXR1cm4gdXgsIHV0LCB1eHgK', 'seed-901-adam-final.pt': '...', 'heat-seed-901.npz': '...', 'seed-902-adam-final.pt': '...', 'heat-seed-902.npz': '...', 'seed-903-adam-final.pt': '...', 'heat-seed-903.npz': '...', 'seed-904-adam-final.pt': '...', 'heat-seed-904.npz': '...', 'seed-905-adam-final.pt': '...', 'heat-seed-905.npz': '...'}
for k, v in BLOB.items(): (p / k).write_bytes(base64.b64decode(v))
n = torch.cuda.device_count(); names = [torch.cuda.get_device_name(i) for i in range(n)]
print(json.dumps({"gpus": names, "hashes": {'heat_lbfgs_uniform_runner.py': '226e882ea4aa0ee11100c5845239902aaa43002484fa69fa4ca23000654f6734', 'af_sciml_model.py': '934687a265c2b2cae0c375cc30365e2d52e0b1d3edfff7cd977db0b59e4d7767', 'seed-901-adam-final.pt': '276f333d6c53fbef174a5eb612ae64cb72a728c9ac6dfc237bb697efa58535f3', 'heat-seed-901.npz': 'c0901df4a0aaad05656bfa779297c77774590b393e06156b6cdd300a0d425213', 'seed-902-adam-final.pt': '093cf763e976ea930a09acef314cc3396f1b8e3fcb671812bf601d8e4a6a465a', 'heat-seed-902.npz': '357844a4bee6599a327f0c44eef116b30be51070342a9286f29f6c5a63e8a314', 'seed-903-adam-final.pt': '9c00d1c392672ae2e2b466dadbf98e7e01ad12f05f56be4128f5ff8582a263d7', 'heat-seed-903.npz': '3eb8aed4319f1c96f91049c3429020bc5f25c820f7440f4a94589a7f5296c5da', 'seed-904-adam-final.pt': '9453b9c0f7a0d54a03c79df4b73d16836829c2d304656c058f8eb26fae98bace', 'heat-seed-904.npz': 'cec4e6613f3154cdb78b495cb0d0f76caccbb2ef74375a23913f5e0a1a21e291', 'seed-905-adam-final.pt': 'dc5a51635d145a4924eca42a69cd28800593a678622469c43d059a82a7d3549c', 'heat-seed-905.npz': '6b9e64f1631befd801be0bdc320c6498396d4d6fb9dbaec8b5f4b07a55d66013'}}), flush=True)
assert n >= 1 and all("T4" in x for x in names)
queue = [901, 902, 903, 904, 905]; running = {}; done = []
while queue or running:
    for g in list(running):
        pr, s, st = running[g]
        if pr.poll() is not None: done.append({"seed": s, "rc": pr.returncode, "wall": time.monotonic() - st}); del running[g]
    if time.monotonic() - T0 > CAP:
        for g, (pr, s, st) in running.items(): pr.kill()
        queue = []; done.append({"note": "cap reached"}); break
    for g in range(min(n, 2)):
        if g not in running and queue:
            s = queue.pop(0); env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
            cmd = [sys.executable, str(p / "heat_lbfgs_uniform_runner.py"), str(s), "--seconds", str(PER_SEED), "--dir", str(p), "--out", str(w / "results"), "--device", "cuda:0"]
            running[g] = (subprocess.Popen(cmd, stdout=open(w / f"seed-{s}.log", "w"), stderr=subprocess.STDOUT, env=env), s, time.monotonic())
    time.sleep(3)
summ = {"done": done, "wall": time.monotonic() - T0, "cap": CAP, "scientific_accepted": False, "reference_accepted": False}
(w / "session_summary.json").write_text(json.dumps(summ, indent=1)); print(json.dumps(summ))
